# Fullscale C / P3_SF / R7_SC — crop1.5

Full eligible official CelebA Train → full official Test-as-Val checkpoint selection and threshold calibration → LCC train/dev/eval/Combined. CelebA Test is reused for selection, not an untouched independent test. No sampling.

Shared controls and exact mechanisms inherit notebook17 / notebooks13–14. One seed; descriptive confirmatory comparison with the frozen design. Set RUN_FILTER to any subset of C_FULL_crop15, P3_SF_FULL_crop15, R7_SC_FULL_crop15. GPU required; no detector inference or resource generation.


In [ ]:
# Edit Kaggle paths here. Full official split manifests are in RESOURCE_ROOT.
RESOURCE_ROOT='/kaggle/input/datasets/maithanhphuong/full-resources-v3/minifasnet_binary_crossdomain_resources_v2_lcc_pooled'
CELEBA_ROOT='/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof'
LCC_ROOT='/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD'
MINIFASNET_SOURCE_PATH='/kaggle/input/datasets/maithanhphuong/minifasnet/MiniFASNet.py'
MINIFASNET_CHECKPOINT_PATH='/kaggle/input/datasets/maithanhphuong/minifasnet/2.7_80x80_MiniFASNetV2.pth'
OUTPUT_ROOT='/kaggle/working/18_fullscale_C_P3SF_R7SC_crop15'
RUN_FILTER=None  # Or any valid subset of the three exact run IDs below.
RESUME=True
NUM_WORKERS=4
AMP=True
DATA_SEED=43
TRAIN_SEED=100
BATCH_SIZE=256
MAX_EPOCHS=12
MILESTONES=[4,8]
GAMMA=.2
PATIENCE_START_EPOCH=9
EARLY_STOP_PATIENCE=3
WARMUP_EPOCHS=1
CSMR_PRESERVE_DC=True
SELECTION_CHUNK_SIZE=32
CROP_SCALE=1.5
TRAIN_CROP_FACTOR=1.5
EVAL_CROP_FACTOR=1.5
LOG_EVERY_BATCHES=25
INPUT_SIZE=80
METRIC_TIE_EPS=1e-12

SCALED_CENTERS=[c*(1.5/2.7) for c in (.15,.45,.75)]
P3_SF_SIGMA=.10*(1.5/2.7)
R7_SC_SIGMA=.10
RUNS={
 'C_FULL_crop15':{'method':'CLEAN','warmup_epochs':0,'crop_factor':1.5,'centers':None,'sigma':None,'clean_ce_weight':1.,'spectral_ce_weight':0.},
 'P3_SF_FULL_crop15':{'method':'WORST','warmup_epochs':1,'crop_factor':1.5,'centers':SCALED_CENTERS,'sigma':P3_SF_SIGMA,'clean_ce_weight':.75,'spectral_ce_weight':.25},
 'R7_SC_FULL_crop15':{'method':'HARMFUL','warmup_epochs':1,'crop_factor':1.5,'centers':SCALED_CENTERS,'sigma':R7_SC_SIGMA,'clean_ce_weight':.75,'spectral_ce_weight':.25},
}
def select_runs(run_filter):
    if run_filter is not None and (not isinstance(run_filter,(list,tuple)) or not run_filter or len(set(run_filter))!=len(run_filter) or set(run_filter)-set(RUNS)):
        raise ValueError('RUN_FILTER must be None or a non-empty unique subset of C_FULL_crop15, P3_SF_FULL_crop15, R7_SC_FULL_crop15')
    return [name for name in RUNS if run_filter is None or name in run_filter]


In [ ]:

import hashlib, time, shutil, importlib.util, inspect, json, math, os, random, zipfile
from collections import Counter, OrderedDict
from pathlib import Path
import albumentations as A
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import Dataset, DataLoader
from IPython.display import FileLink, display

_BBOX_RNG=random.Random(TRAIN_SEED+500000)
AUG_SEED=TRAIN_SEED
BBOX_SEED=TRAIN_SEED+500000
GAIN_SEED=TRAIN_SEED+11000
GAIN_RANGE=(.65,.90)
SPLITS=('training','development','evaluation')
EXPECTED_LCC={'training':8299,'development':2948,'evaluation':7580,'combined':18827}
if RUN_FILTER is not None and (not isinstance(RUN_FILTER,(list,tuple)) or not RUN_FILTER or len(set(RUN_FILTER))!=len(RUN_FILTER) or set(RUN_FILTER)-set(RUNS)):
    raise ValueError('RUN_FILTER contains an unknown or duplicate run ID')
if NUM_WORKERS<1:raise ValueError('NUM_WORKERS must be >= 1')
if (BATCH_SIZE,MAX_EPOCHS,MILESTONES,GAMMA,TRAIN_SEED)!=(256,12,[4,8],.2,100):raise RuntimeError('Frozen fullscale recipe changed')
root=Path(RESOURCE_ROOT);out=Path(OUTPUT_ROOT);source=Path(MINIFASNET_SOURCE_PATH);pretrained=Path(MINIFASNET_CHECKPOINT_PATH)
for path in (source,pretrained,root/'manifests/crossdomain_train_full.csv',
             root/'manifests/crossdomain_celeba_source_dev_full.csv',root/'bbox/celeba_bbox_cache_full.json',
             root/'labels/celeba_auxiliary_labels.csv'):
    if not path.is_file():raise FileNotFoundError(path)
def read_json(path):return json.loads(Path(path).read_text())
def write_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,sort_keys=True,allow_nan=False)+'\n')
def save_torch(path,value):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp')
    torch.save(value,tmp);os.replace(tmp,path)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1048576),b''): h.update(block)
    return h.hexdigest()
def state_sha(state):
    h=hashlib.sha256()
    for k in sorted(state):
        v=state[k].detach().cpu().contiguous();h.update(k.encode());h.update(str(v.dtype).encode());h.update(str(tuple(v.shape)).encode());h.update(v.numpy().tobytes())
    return h.hexdigest()
def require_record(rec,key):
    if rec.get('status')!='VALID' or rec.get('usable_for_pad') is not True or rec.get('bbox_origin')!='SCRFD' or rec.get('source_key')!=key or not str(rec.get('bbox_source','')).startswith('SCRFD'):raise RuntimeError('Unscorable/non-SCRFD: '+key)
    box=rec.get('bbox_xyxy')
    if box is None or len(box)!=4 or 'crop_factor' in rec:raise RuntimeError('Not a raw bbox: '+key)
    x1,y1,x2,y2=map(float,box)
    if not np.isfinite(box).all() or x2<=x1 or y2<=y1:raise RuntimeError('Invalid bbox: '+key)
    return rec

def mini_crop_bgr(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE):
    # Exact 03b CropImage-compatible geometry.
    src_h,src_w=image.shape[:2];x1,y1,x2,y2=map(float,bbox_xyxy);box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:raise ValueError('Invalid crop geometry')
    used=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale));new_w,new_h=box_w*used,box_h*used
    cx,cy=box_w/2+x1,box_h/2+y1;left,top=cx-new_w/2,cy-new_h/2;right,bottom=cx+new_w/2,cy+new_h/2
    if left<0:right-=left;left=0
    if top<0:bottom-=top;top=0
    if right>src_w-1:left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1:top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:raise RuntimeError('Crop escaped image')
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty crop')
    return cv2.resize(patch,(size,size))

# The literature's augmentation ranges are fixed. Support both documented Albumentations APIs.
# ISONoise works in RGB/HSV; convert only within augmentation, then restore BGR for MiniFASNet.
iso_params=inspect.signature(A.ISONoise).parameters
brightness_params=inspect.signature(A.RandomBrightnessContrast).parameters
blur_params=inspect.signature(A.MotionBlur).parameters
if {'color_shift','intensity'}<=set(iso_params) and {'brightness_limit','contrast_limit'}<=set(brightness_params) and 'blur_limit' in blur_params:
    AUGMENTATION_API='legacy_limits'
    iso=A.ISONoise(color_shift=(.15,.35),intensity=(.2,.5),p=.2)
    bright=A.RandomBrightnessContrast(brightness_limit=.2,contrast_limit=.2,brightness_by_max=True,p=.3)
    motion=A.MotionBlur(blur_limit=5,p=.2)
elif {'color_shift_range','intensity_range'}<=set(iso_params) and {'brightness_range','contrast_range'}<=set(brightness_params) and 'blur_range' in blur_params:
    AUGMENTATION_API='range_parameters'
    iso=A.ISONoise(color_shift_range=(.15,.35),intensity_range=(.2,.5),p=.2)
    bright=A.RandomBrightnessContrast(brightness_range=(-.2,.2),contrast_range=(-.2,.2),brightness_by_max=True,p=.3)
    motion=A.MotionBlur(blur_range=(3,5),p=.2)
else:raise RuntimeError('Unsupported Albumentations augmentation API; freeze and audit a compatible version')
COPY_AUG=A.Compose([A.HorizontalFlip(p=.5),iso,bright,motion])
def copied_aug_bgr(patch):
    rgb=cv2.cvtColor(patch,cv2.COLOR_BGR2RGB)
    augmented=COPY_AUG(image=rgb)['image']
    return cv2.cvtColor(augmented,cv2.COLOR_RGB2BGR)
def tensor_of(patch):
    if patch.shape!=(80,80,3) or patch.dtype!=np.uint8:raise ValueError('Expected BGR uint8 80x80')
    return torch.from_numpy(np.ascontiguousarray(patch.transpose(2,0,1))).float().div_(255.)

class PADNet(nn.Module):
    def __init__(self,copied):
        super().__init__();self.copied=copied
        self.trunk=module.MiniFASNetV2(embedding_size=128,conv6_kernel=(5,5),drop_p=.2,num_classes=3,img_channel=3)
        if copied:
            self.trunk.prob=nn.Identity()
            self.pad_head=nn.Linear(128,2);self.spoof_type_head=nn.Linear(128,11)
            self.lighting_head=nn.Linear(128,5);self.attributes_head=nn.Linear(128,40)
    def features(self,x):
        t=self.trunk
        for layer in (t.conv1,t.conv2_dw,t.conv_23,t.conv_3,t.conv_34,t.conv_4,t.conv_45,t.conv_5,t.conv_6_sep,t.conv_6_dw,t.conv_6_flatten):x=layer(x)
        if t.embedding_size!=512:x=t.linear(x)
        x=t.drop(t.bn(x))
        if x.ndim!=2 or x.shape[1]!=128:raise RuntimeError('MiniFAS representation is not [B,128]')
        return x
    def forward(self,x,aux=False):
        z=self.features(x)
        if not self.copied:return self.trunk.prob(z)
        pad=self.pad_head(z)
        if aux:return pad,self.spoof_type_head(z),self.lighting_head(z),self.attributes_head(z)
        return pad

def pad_score(logits):
    z=torch.as_tensor(logits)
    if z.shape[1]==3:return (z[:,0]-torch.logsumexp(z[:,1:],dim=1)).float().cpu().numpy()
    if z.shape[1]==2:return (z[:,0]-z[:,1]).float().cpu().numpy()
    raise ValueError('Expected two or three PAD logits')
def threshold_from_val(y3,scores):
    y=np.asarray(y3,dtype=np.int64);s=np.asarray(scores,dtype=np.float64)
    if len(y)!=len(s) or not len(y) or not np.isfinite(s).all() or set(np.unique(y))!={0,1,2}:raise ValueError('Val needs all three classes')
    order=np.argsort(s,kind='mergesort');v=s[order];real=(y[order]==0);nr=int(real.sum());na=len(y)-nr
    bound=np.r_[0,np.flatnonzero(np.diff(v))+1,len(v)];rp=np.r_[0,np.cumsum(real)];ap=np.r_[0,np.cumsum(~real)];pairs=[]
    for j,k in enumerate(bound):
        t=float(np.nextafter(v[0],-np.inf)) if j==0 else (float(np.nextafter(v[-1],np.inf)) if j==len(bound)-1 else float(v[k-1]/2+v[k]/2))
        pairs.append((.5*(rp[k]/nr+(na-ap[k])/na),abs(t),t))
    return min(pairs)[2]
def metrics(y3,scores,threshold,candidate_n):
    y=np.asarray(y3,dtype=int);s=np.asarray(scores,dtype=float);real=y==0
    if len(y)!=len(s) or set(real)!={False,True} or not np.isfinite(s).all():raise ValueError('Invalid metric population')
    fpr,tpr,thr=roc_curve(real.astype(int),s);eer_idx=int(np.argmin(abs(fpr-(1-tpr))))
    pred=s>=threshold;apcer=float(pred[~real].mean());bpcer=float((~pred[real]).mean())
    return {'auc':float(roc_auc_score(real.astype(int),s)),'eer':float(.5*(fpr[eer_idx]+1-tpr[eer_idx])),
      'tpr_at_fpr_1pct':float(np.interp(.01,fpr,tpr)),'apcer':apcer,'bpcer':bpcer,'acer':.5*(apcer+bpcer),
      'hter':.5*(apcer+bpcer),'binary_accuracy':float((pred==real).mean()),'locked_source_threshold':float(threshold),
      'candidate_n':int(candidate_n),'scored_n':len(y),'detector_coverage':len(y)/candidate_n}

def radial_masks(centers,sigma,size=INPUT_SIZE,device=None):
    if device is None:device=globals()['device']
    # Same radial geometry as the tested 03b CSMR notebook.
    y=torch.arange(size,device=device,dtype=torch.float32)-size//2
    yy,xx=torch.meshgrid(y,y,indexing='ij')
    radius=torch.sqrt(xx.square()+yy.square())/math.sqrt(2*(size//2)**2)
    centers=torch.as_tensor(centers,dtype=torch.float32,device=device)
    masks=torch.exp(-((radius[None]-centers[:,None,None])**2)/(2*sigma**2))
    if masks.shape!=(3,size,size) or not torch.isfinite(masks).all():raise RuntimeError('Invalid CSMR radial masks')
    return masks
def spectral_views(x,gains,masks):
    if x.ndim!=4 or x.shape[1:]!=(3,80,80) or not torch.isfinite(x).all():raise ValueError('CSMR input')
    x=x.float();gains=gains.float().reshape(-1)
    if len(gains)!=len(x) or not ((gains>=0)&(gains<=1)).all():raise ValueError('CSMR gain')
    spectrum=torch.fft.fftshift(torch.fft.fft2(x,norm='ortho'),dim=(-2,-1))
    fields=1-(1-gains[:,None,None,None])*masks[None]
    if CSMR_PRESERVE_DC:fields[:,:,40,40]=1
    raw=torch.fft.ifft2(torch.fft.ifftshift(spectrum[:,None]*fields[:,:,None],dim=(-2,-1)),norm='ortho').real
    clips=((raw<0)|(raw>1)).float().mean(dim=(2,3,4));views=raw.clamp(0,1)
    if not torch.isfinite(views).all():raise RuntimeError('Nonfinite CSMR views')
    return views,clips
def aligned_margin(logits,binary):
    d=logits[:,0].float()-logits[:,1].float()
    return torch.where(binary==0,d,-d)

def auxiliary_loss(outputs,binary,spoof,lighting,attrs):
    pad,spoof_logits,light_logits,attr_logits=outputs
    ls=F.cross_entropy(spoof_logits,spoof);ll=F.cross_entropy(light_logits,lighting);mask=binary==0
    la=F.binary_cross_entropy_with_logits(attr_logits[mask],attrs[mask]) if mask.any() else attr_logits.sum()*0
    return ls,ll,la
def restore_rng(state,gain_rng):
    random.setstate(state['python']);np.random.set_state(state['numpy']);torch.set_rng_state(state['torch'])
    torch.cuda.set_rng_state_all(state['cuda']);gain_rng.set_state(state['gain']);_BBOX_RNG.setstate(state['bbox'])
def bbox_jitter(rec):
    x1,y1,x2,y2=map(float,rec['bbox_xyxy']);w,h=x2-x1,y2-y1
    if _BBOX_RNG.random()<.20:
        base=max(w,h);cx,cy=(x1+x2)/2,(y1+y2)/2;scale=_BBOX_RNG.uniform(.95,1.05)
        cx+=_BBOX_RNG.uniform(-.05,.05)*base;cy+=_BBOX_RNG.uniform(-.05,.05)*base;w*=scale;h*=scale
        return [cx-w/2,cy-h/2,cx+w/2,cy+h/2]
    return [x1,y1,x2,y2]


def load_frame(name):return pd.read_csv(root/'manifests'/(name+'.csv'),keep_default_na=False)
train=load_frame('crossdomain_train_full')
test=load_frame('crossdomain_celeba_source_dev_full')
val=test.loc[test.usable_for_pad.astype(bool)].copy()  # Full official Test-as-Val selection set.
records=read_json(root/'bbox/celeba_bbox_cache_full.json')['records']
aux=pd.read_csv(root/'labels/celeba_auxiliary_labels.csv',keep_default_na=False).set_index('sample_key',verify_integrity=True)
if (len(train),len(test),len(val))!=(484561,67170,66379):raise RuntimeError('Unexpected full official populations')
if train.sample_key.duplicated().any() or test.sample_key.duplicated().any() or set(train.sample_key)&set(test.sample_key):raise RuntimeError('Train/Test sample overlap')
for frame in (train,val,test):
    if not set(frame.sample_key)<=set(records) or not set(frame.sample_key)<=set(aux.index):raise RuntimeError('Missing cache/aux entry')
for frame in (train,val):
    if not frame.usable_for_pad.astype(bool).all() or set(frame.bbox_origin)!={'SCRFD'} or set(frame.three_class_label)!={0,1,2}:raise RuntimeError('Unusable full split')
# Candidate accounting uses existing official metadata, with the unchanged 00c eligibility rule.
def train_exclusion_reason(key):
    rec=records[key]
    if rec['status'] in {'SUSPICIOUS','DET_FAIL','UNREADABLE'}:return rec['status']
    try:require_record(rec,key)
    except RuntimeError:return 'INVALID_BBOX'
    x1,y1,x2,y2=map(float,rec['bbox_xyxy'])
    return 'TOO_SMALL' if min(x2-x1,y2-y1)<48 else ''
train_candidates=aux.loc[aux.official_split=='train'].reset_index()
train_candidates['exclusion_reason']=[train_exclusion_reason(k) for k in train_candidates.sample_key]
if len(train_candidates)!=494405 or set(train_candidates.loc[train_candidates.exclusion_reason=='','sample_key'])!=set(train.sample_key):raise RuntimeError('Full train membership differs from official eligibility; no sampling allowed')
if set(aux.loc[aux.official_split=='test'].index)!=set(test.sample_key):raise RuntimeError('Test manifest is not the full official Test')
def population_counts(frame,three_col='three_class_label',attack_col='attack_code'):
    classes={str(k):int(v) for k,v in frame[three_col].value_counts().sort_index().items()}
    return {'n':len(frame),'three_class_counts':classes,'real_n':classes.get('0',0),'attack_n':len(frame)-classes.get('0',0),
            'spoof_type_counts':{str(k):int(v) for k,v in frame[attack_col].value_counts().sort_index().items()}}
DATA_COUNTS={'train_candidates':population_counts(train_candidates,attack_col='spoof_type'),
 'train_usable':population_counts(train),'train_excluded':len(train_candidates)-len(train),
 'train_exclusion_reasons':{str(k):int(v) for k,v in train_candidates.loc[train_candidates.exclusion_reason!='','exclusion_reason'].value_counts().items()},
 'test_candidates':population_counts(test),'test_usable':population_counts(val),'test_unscored':len(test)-len(val),
 'train_test_identity_overlap':sorted(set(train.subject_id.astype(str))&set(test.subject_id.astype(str))),
 'identity_overlap_policy':'Official membership retained; no subject removal or new split',
 'sampling':'None; full official eligible Train and full official Test-as-Val', 'data_seed':DATA_SEED}
lcc_frames={};lcc_records={}
for split in ('training','development','evaluation','combined'):
    name='lcc_official_evaluation_full' if split=='evaluation' else 'lcc_'+split+'_full'
    cache_name='lcc_official_evaluation_bbox_cache' if split=='evaluation' else 'lcc_'+split+'_bbox_cache'
    path=root/'manifests'/(name+'.csv');cache=root/'bbox'/(cache_name+'.json')
    if not path.is_file() or not cache.is_file():raise FileNotFoundError((path,cache))
    frame=pd.read_csv(path,keep_default_na=False)
    if len(frame)!=EXPECTED_LCC[split] or frame.path.duplicated().any():raise RuntimeError('Unexpected LCC population: '+split)
    recs=read_json(cache)['records'] if split!='combined' else None
    if recs is not None and set(frame.path)!=set(recs):raise RuntimeError('LCC manifest/cache keys differ: '+split)
    lcc_frames[split]=frame;lcc_records[split]=recs
ordered_pool=pd.concat([lcc_frames[s].assign(lcc_split=s) for s in SPLITS],ignore_index=True)
if not lcc_frames['combined'].equals(ordered_pool):raise RuntimeError('Combined LCC manifest differs from physical splits')
if not torch.cuda.is_available():raise RuntimeError('Kaggle CUDA GPU required for the frozen fullscale batch size')
source_spec=importlib.util.spec_from_file_location('local_minifasnet_v2',source)
module=importlib.util.module_from_spec(source_spec);source_spec.loader.exec_module(module)
device=torch.device('cuda');use_amp=bool(AMP)
out.mkdir(parents=True,exist_ok=True);(out/'runs').mkdir(exist_ok=True)
(out/'manifests').mkdir(exist_ok=True)
MANIFEST_SOURCES={'full_train.csv':root/'manifests/crossdomain_train_full.csv',
                  'full_test.csv':root/'manifests/crossdomain_celeba_source_dev_full.csv'}
MANIFEST_HASHES={f:sha(p) for f,p in MANIFEST_SOURCES.items()}
for filename,path in MANIFEST_SOURCES.items():shutil.copyfile(path,out/'manifests'/filename)
train_candidates.to_csv(out/'manifests'/'full_train_candidates.csv',index=False)
write_json(out/'manifest_sha256.json',MANIFEST_HASHES)
write_json(out/'dataset_counts.json',DATA_COUNTS)
print('GPU:',torch.cuda.get_device_name(0),'Full Train:',len(train),'Full Test-as-Val usable/candidate:',len(val),len(test),'LCC:',{s:len(f) for s,f in lcc_frames.items()},flush=True)

def init_model(spec):
    seed_all(TRAIN_SEED);model=PADNet(spec['copied']);audit={'initialization':spec['init'],'checkpoint_sha256':sha(pretrained),'source_sha256':sha(source)}
    if spec['init']=='pretrained':
        raw=torch.load(pretrained,map_location='cpu',weights_only=True)
        if not isinstance(raw,OrderedDict) or not raw or not all(k.startswith('module.') for k in raw):raise RuntimeError('Official checkpoint format mismatch')
        stripped=OrderedDict((k[7:],v) for k,v in raw.items())
        if tuple(stripped['prob.weight'].shape)!=(3,128) or tuple(stripped['conv_6_dw.conv.weight'].shape[-2:])!=(5,5):raise RuntimeError('Official checkpoint architecture mismatch')
        target=model.trunk.state_dict();compatible={k:v for k,v in stripped.items() if k!='prob.weight'}
        bad=[k for k,v in compatible.items() if k not in target or tuple(v.shape)!=tuple(target[k].shape)]
        absent=[k for k in target if k not in compatible and k!='prob.weight']
        if bad or absent:raise RuntimeError(f'Pretrained key mismatch: {bad[:5]} {absent[:5]}')
        missing,unexpected=model.trunk.load_state_dict(compatible,strict=False)
        if unexpected or (spec['copied'] and missing) or (not spec['copied'] and set(missing)!={'prob.weight'}):raise RuntimeError('Pretrained load mismatch')
        # Reinitialize historical classifier too; copied heads were initialized by seed 100.
        if not spec['copied']:model.trunk.prob.reset_parameters()
        audit.update({'loaded_keys':sorted(compatible),'skipped_keys':['prob.weight'],'missing_keys':list(missing),'unexpected_keys':list(unexpected)})
    else:audit.update({'loaded_keys':[],'skipped_keys':[],'missing_keys':[],'unexpected_keys':[]})
    audit['initial_state_sha256']=state_sha(model.state_dict())
    with torch.no_grad():
        model.eval();assert model(torch.zeros(2,3,80,80)).shape==(2,2 if spec['copied'] else 3)
    return model,audit

INITIAL_AUDITS={}
for name in RUNS:
    initial_model,initial_audit=init_model({'init':'pretrained','copied':True})
    INITIAL_AUDITS[name]=initial_audit
    del initial_model
INITIAL_HASHES={name:audit['initial_state_sha256'] for name,audit in INITIAL_AUDITS.items()}
if len(set(INITIAL_HASHES.values()))!=1:raise RuntimeError('Initial-state SHA differs across all three methods')
INITIAL_STATE_SHA=next(iter(INITIAL_HASHES.values()))
write_json(out/'initialization_fairness.json',INITIAL_AUDITS)

class CelebaData(Dataset):
    def __init__(self,frame,training=False,copied=False):self.rows=frame.reset_index(drop=True);self.training=training;self.copied=copied
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.sample_key);rec=require_record(records[key],key)
        image=cv2.imread(str(Path(CELEBA_ROOT)/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(key)
        patch=mini_crop_bgr(image,bbox_jitter(rec) if self.training else rec['bbox_xyxy'])
        if self.training:patch=copied_aug_bgr(patch) if self.copied else mild_aug(patch)
        a=aux.loc[key]
        return tensor_of(patch),int(row.three_class_label),int(row.binary_label),int(a.spoof_type),int(a.lighting),torch.tensor([float(a[f'attr_{j:02d}']) for j in range(40)]),key

class LCCData(Dataset):
    def __init__(self,split):
        self.split=split;self.rows=lcc_frames[split].loc[lcc_frames[split].usable_for_pad.astype(bool)].reset_index(drop=True)
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.path);rec=require_record(lcc_records[self.split][key],key)
        path=(Path(LCC_ROOT)/'LCC_FASD_evaluation'/key) if self.split=='evaluation' else Path(LCC_ROOT)/key
        image=cv2.imread(str(path),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(path)
        return tensor_of(mini_crop_bgr(image,rec['bbox_xyxy'])),int(row.label),key

def loader(ds,batch=BATCH_SIZE,shuffle=False,epoch=0):
    gen=torch.Generator().manual_seed(TRAIN_SEED+epoch)
    def worker_init(wid):
        global _BBOX_RNG
        s=TRAIN_SEED+epoch*1000+wid;random.seed(s);np.random.seed(s);torch.manual_seed(s)
        _BBOX_RNG=random.Random(s+500000)
        if hasattr(COPY_AUG,'set_random_seed'):COPY_AUG.set_random_seed(s)
    return DataLoader(ds,batch_size=batch,shuffle=shuffle,num_workers=NUM_WORKERS,pin_memory=True,drop_last=False,generator=gen,worker_init_fn=worker_init)

def infer(model,dl,lcc_mode=False):
    model.eval();keys=[];ys=[];zs=[];started=time.monotonic()
    print('[Inference] Starting',len(dl.dataset),'scored images',flush=True)
    with torch.no_grad():
        for step,batch in enumerate(dl,1):
            if lcc_mode:x,y,k=batch
            else:x,y,_,_,_,_,k=batch
            with torch.cuda.amp.autocast(enabled=use_amp):z=model(x.to(device,non_blocking=True))
            keys.extend(k);ys.extend(y.numpy().tolist());zs.append(z.float().cpu().numpy())
            if step==1 or step%LOG_EVERY_BATCHES==0 or step==len(dl):print(f'[Inference] batch {step}/{len(dl)}, elapsed={time.monotonic()-started:.1f}s',flush=True)
    if not zs:raise RuntimeError('No scorable evaluation samples')
    return list(map(str,keys)),np.asarray(ys,dtype=int),np.concatenate(zs)

def select_worst_view(model,base,binary,masks,gain_rng):
    was_training=model.training;model.eval()
    chosen=[];bands=[];gains_out=[];clips_out=[];clean_margins=[];worst_margins=[]
    try:
        with torch.no_grad():
            all_gains=torch.empty(len(base),device=base.device).uniform_(*GAIN_RANGE,generator=gain_rng)
            for start in range(0,len(base),SELECTION_CHUNK_SIZE):
                x=base[start:start+SELECTION_CHUNK_SIZE];y=binary[start:start+SELECTION_CHUNK_SIZE]
                gains=all_gains[start:start+len(x)]
                with torch.cuda.amp.autocast(enabled=use_amp):clean_z=model(x)
                clean_margins.append(aligned_margin(clean_z,y).detach())
                views,clips=spectral_views(x,gains,masks);margins=[]
                for band in range(3):
                    with torch.cuda.amp.autocast(enabled=use_amp):z=model(views[:,band])
                    margins.append(aligned_margin(z,y))
                margin=torch.stack(margins,dim=1);ix=margin.argmin(dim=1)
                row=torch.arange(len(x),device=x.device)
                chosen.append(views[row,ix].detach());bands.append(ix.detach())
                gains_out.append(gains.detach());clips_out.append(clips[row,ix].detach())
                worst_margins.append(margin[row,ix].detach())
    finally:model.train(was_training)
    return tuple(torch.cat(seq) for seq in (chosen,bands,gains_out,clips_out,clean_margins,worst_margins))


def training_terms(model,x,binary,spoof,lighting,attrs,spec,masks,gain_rng,active):
    method=spec['method'];n=len(x);selected=None;bands=None;gains=None;select_clean=None;select_worst=None
    spectral_active=active and method!='CLEAN'
    if spectral_active:
        selected,bands,gains,_,select_clean,select_worst=select_worst_view(model,x,binary,masks,gain_rng)
    with torch.cuda.amp.autocast(enabled=use_amp):
        clean=model(x,aux=True);clean_ce=F.cross_entropy(clean[0],binary)
        ls,ll,la=auxiliary_loss(clean,binary,spoof,lighting,attrs);aux_total=.1*ls+.1*ll+la
        spectral_ce=clean_ce*0;worst_ce=clean_ce*0;harmful_count=0;spectral_logits=None;harmful_ce_sum=0.
        if spectral_active:
            spectral_logits=model(selected);ce_each=F.cross_entropy(spectral_logits,binary,reduction='none');worst_ce=ce_each.mean()
            if method=='HARMFUL':
                harmful=(select_worst<select_clean).detach();harmful_count=int(harmful.sum())
                harmful_ce_sum=float(ce_each.detach()[harmful].sum())
                spectral_ce=(ce_each*harmful).sum()/harmful_count if harmful_count else spectral_logits.sum()*0
            else:spectral_ce=worst_ce
        loss=.75*clean_ce+.25*spectral_ce+aux_total if spectral_active else clean_ce+aux_total
    with torch.no_grad():
        cm=aligned_margin(clean[0],binary).mean()
        flips=int(((clean[0].argmax(1)==binary)&(spectral_logits.argmax(1)!=binary)).sum()) if spectral_logits is not None else 0
        scalars={'train_total_loss':float(loss.detach()),'total_loss':float(loss.detach()),'clean_ce':float(clean_ce.detach()),
          'spectral_ce':float(spectral_ce.detach()),'worst_ce':float(worst_ce.detach()),'aux_loss':float(aux_total.detach()),
          'spoof_type_ce':float(ls.detach()),'lighting_ce':float(ll.detach()),'attributes_real_bce':float(la.detach()),
          'clean_aligned_margin':float(select_clean.mean()) if select_clean is not None else float(cm),
          'clean_margin':float(select_clean.mean()) if select_clean is not None else float(cm),
          'train_clean_aligned_margin':float(cm),'worst_margin':float(select_worst.mean()) if select_worst is not None else 0.,
          'margin_drop':float((select_clean-select_worst).mean()) if select_worst is not None else 0.,
          'spectral_flip_rate':flips/n,'harmful_count':harmful_count,'harmful_fraction':harmful_count/n,
          'harmful_ce':float(spectral_ce.detach()) if method=='HARMFUL' and spectral_active else 0.,
          'sampled_gain_mean':float(gains.mean()) if gains is not None else 0.}
    counts=Counter(bands.detach().cpu().tolist()) if bands is not None else Counter()
    return loss,scalars,counts,harmful_ce_sum

def snapshot_rng(gain_rng):
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
            'cuda':torch.cuda.get_rng_state_all(),'gain':gain_rng.get_state(),'bbox':_BBOX_RNG.getstate()}

def run_config(name,spec):
    return {'run_id':name,'spec':spec,'resource_root':str(root),'train_manifest':'full_train.csv',
      'val_manifest':'full_test.csv','selection_set':'Full official CelebA Test-as-Val; reused selection set, not untouched Test','manifest_sha256':MANIFEST_HASHES,
      'train_crop_factor':1.5,'val_crop_factor':1.5,'test_crop_factor':1.5,'lcc_crop_factor':1.5,'data_seed':DATA_SEED,
      'train_seed':TRAIN_SEED,'initial_state_sha256':INITIAL_STATE_SHA,
      'base_order':'torch Generator(TRAIN_SEED+epoch)','worker_seed':'TRAIN_SEED+epoch*1000+worker_id',
      'bbox_rng_seed':'worker_seed+500000','gain_rng_seed':GAIN_SEED,
      'pretrained_checkpoint':str(pretrained),'model_source':str(source),
      'celeba_root':CELEBA_ROOT,'lcc_root':LCC_ROOT,'num_workers':NUM_WORKERS,
      'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS,'milestones':MILESTONES,'gamma':GAMMA,
      'optimizer':{'name':'SGD','lr':.005,'momentum':.9,'weight_decay':5e-4},
      'warmup_epochs':spec['warmup_epochs'],'patience_start_epoch':PATIENCE_START_EPOCH,'early_stop_patience':EARLY_STOP_PATIENCE,
      'spectral':None if spec['method']=='CLEAN' else {'centers':spec['centers'],'sigma':spec['sigma'],'gain':[.65,.90],'preserve_dc':CSMR_PRESERVE_DC,'selection_chunk_size':SELECTION_CHUNK_SIZE},
      'input':'80x80 BGR float [0,1] crop1.5','augmentation_api':AUGMENTATION_API,
      'albumentations_version':A.__version__,'amp':use_amp}

def prediction_frame(frame,keys,y,z,threshold,keycol):
    scores=pad_score(z)
    scored=pd.DataFrame({keycol:keys,'pad_logit_score':scores,'pred_real':scores>=threshold,
                         'logit_0':z[:,0],'logit_1':z[:,1]})
    result=frame.merge(scored,on=keycol,how='left',validate='one_to_one',sort=False)
    valid=frame.usable_for_pad.astype(bool)
    if result.loc[valid,'pad_logit_score'].isna().any() or result.loc[~valid,'pad_logit_score'].notna().any():
        raise RuntimeError('Prediction accounting differs from frozen candidates')
    result['locked_threshold']=threshold
    return result

def evaluate_lcc(model,folder,locked):
    scored_predictions=[]
    for split in SPLITS:
        frame=lcc_frames[split]
        keys,y,z=infer(model,loader(LCCData(split)),True)
        expected=frame.loc[frame.usable_for_pad.astype(bool),'path'].astype(str).tolist()
        if keys!=expected:raise RuntimeError('LCC scored order differs: '+split)
        pred=prediction_frame(frame,keys,y,z,locked,'path').assign(lcc_split=split)
        mm=metrics(y,pad_score(z),locked,len(frame))
        write_json(folder/'metrics'/f'lcc_{split}_metrics.json',mm)
        write_json(folder/'metrics'/f'lcc_{split}_reference_metrics.json',reference_metrics(y,z,len(frame)))
        pred.to_csv(folder/'predictions'/f'lcc_{split}_predictions.csv',index=False)
        scored_predictions.append(pred)
    pooled=pd.concat(scored_predictions,ignore_index=True)
    if not pooled[['path','label','status','usable_for_pad','bbox_origin','lcc_split']].equals(
        lcc_frames['combined'][['path','label','status','usable_for_pad','bbox_origin','lcc_split']]):
        raise RuntimeError('Combined prediction candidates differ from 00d manifest')
    usable=pooled.usable_for_pad.astype(bool)
    mm=metrics(pooled.loc[usable,'label'],pooled.loc[usable,'pad_logit_score'],locked,len(pooled))
    write_json(folder/'metrics'/'lcc_combined_metrics.json',mm)
    write_json(folder/'metrics'/'lcc_combined_reference_metrics.json',reference_metrics(pooled.loc[usable,'label'],pooled.loc[usable,['logit_0','logit_1']].to_numpy(),len(pooled)))
    pooled.to_csv(folder/'predictions'/'lcc_combined_predictions.csv',index=False)

def save_final_source_evaluation(folder,keys,y,z,locked):
    mm=metrics(y,pad_score(z),locked,len(test))
    write_json(folder/'metrics'/'test_metrics.json',mm)
    write_json(folder/'metrics'/'test_reference_metrics.json',reference_metrics(y,z,len(test)))
    prediction_frame(test,keys,y,z,locked,'sample_key').to_csv(folder/'predictions'/'test_predictions.csv',index=False)

def run_one(name,spec):
    folder=out/'runs'/name;folder.mkdir(parents=True,exist_ok=True);(folder/'predictions').mkdir(exist_ok=True);(folder/'metrics').mkdir(exist_ok=True)
    cfg=run_config(name,spec);cfg_path=folder/'config.json'
    if cfg_path.is_file() and read_json(cfg_path)!=cfg:raise RuntimeError(name+' existing config differs')
    write_json(cfg_path,cfg)
    (folder/'manifests').mkdir(exist_ok=True)
    for filename in MANIFEST_SOURCES:shutil.copyfile(out/'manifests'/filename,folder/'manifests'/filename)
    best_path=folder/'best.pth';last_path=folder/'last_state.pth';done_path=folder/'completion.json'
    if done_path.is_file():
        needed=[folder/'metrics'/(k+'_metrics.json') for k in ('val','test','lcc_training','lcc_development','lcc_evaluation','lcc_combined')]
        if not all(p.is_file() for p in needed) or not best_path.is_file() or not (folder/'evaluation_freeze.json').is_file() or not all((folder/'metrics'/(s+'_reference_metrics.json')).is_file() for s in ('test','lcc_training','lcc_development','lcc_evaluation','lcc_combined')):
            raise RuntimeError(name+' incomplete completion outputs')
        print(name,'completed; skipping')
        return
    model,audit=init_model({'init':'pretrained','copied':True})
    if audit['initial_state_sha256']!=INITIAL_STATE_SHA:raise RuntimeError('Initial state differs')
    write_json(folder/'initialization_audit.json',audit)
    model=model.to(device)
    optimizer=torch.optim.SGD(model.parameters(),lr=.005,momentum=.9,weight_decay=5e-4)
    scheduler=torch.optim.lr_scheduler.MultiStepLR(optimizer,milestones=MILESTONES,gamma=GAMMA)
    scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
    gain_rng=torch.Generator(device=device).manual_seed(GAIN_SEED)
    best_checkpoint=None
    history=[];diagnostics=[];best_acer=float('inf');best_auc=-float('inf');best_epoch=0;no_improve=0;start=1
    if last_path.is_file():
        if not RESUME:raise RuntimeError(name+' has a saved state; enable RESUME or use a new output root')
        state=torch.load(last_path,map_location='cpu',weights_only=False)
        if state.get('run_id')!=name or state.get('config')!=cfg:raise RuntimeError(name+' resume run/config differs')
        if state.get('initial_state_sha256')!=INITIAL_STATE_SHA:raise RuntimeError('Resume initial state differs')
        best_checkpoint=state['best_checkpoint']
        if state['best_epoch']:
            if best_checkpoint['run_id']!=name or best_checkpoint['epoch']!=state['best_epoch'] or best_checkpoint['config']!=cfg:raise RuntimeError('Resume best payload differs')
            save_torch(best_path,best_checkpoint)
        model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
        history=state['history'];diagnostics=state['diagnostics'];best_acer=state['best_acer']
        best_auc=state['best_auc'];best_epoch=state['best_epoch'];no_improve=state['no_improve']
        start=state['epoch']+1
        if len(history)!=state['epoch']:raise RuntimeError(name+' resume history length differs')
        restore_rng(state['rng'],gain_rng)
        print(name,'resume at epoch',start)
    val_dl=loader(CelebaData(val))
    masks=radial_masks(spec['centers'],spec['sigma']) if spec['method']!='CLEAN' else None
    if start<=MAX_EPOCHS and not (history and history[-1]['stop_reached']):
        for epoch in range(start,MAX_EPOCHS+1):
            expected_lr=.005 if epoch<=4 else (.001 if epoch<=8 else .0002)
            if not math.isclose(optimizer.param_groups[0]['lr'],expected_lr,rel_tol=1e-9):raise RuntimeError('Scheduler LR differs')
            warmup=epoch<=spec['warmup_epochs']
            active=not warmup
            model.train();totals=Counter();band_count=Counter();seen=0;harmful_count_total=0;harmful_ce_sum_total=0.;epoch_started=time.monotonic()
            train_ds=CelebaData(train,training=True,copied=True)
            train_dl=loader(train_ds,shuffle=True,epoch=epoch)
            print(f'[{name}] epoch {epoch}/{MAX_EPOCHS} starting, warmup={warmup}',flush=True)
            for step,(x,_,binary,spoof,lighting,attrs,_) in enumerate(train_dl,1):
                x=x.to(device,non_blocking=True);binary=binary.to(device);spoof=spoof.to(device)
                lighting=lighting.to(device);attrs=attrs.to(device);optimizer.zero_grad(set_to_none=True)
                loss,scalars,bands,harmful_ce_sum=training_terms(model,x,binary,spoof,lighting,attrs,
                    spec,masks,gain_rng,active)
                scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()
                count=len(x);seen+=count
                for key,value in scalars.items():totals[key]+=value*count
                band_count.update(bands)
                harmful_count_total+=scalars['harmful_count'];harmful_ce_sum_total+=harmful_ce_sum
                if step==1 or step%LOG_EVERY_BATCHES==0 or step==len(train_dl):
                    print(f'[{name}] epoch {epoch} batch {step}/{len(train_dl)}, loss={totals["train_total_loss"]/seen:.4f}, elapsed={time.monotonic()-epoch_started:.1f}s',flush=True)
            keys,y,z=infer(model,val_dl)
            if keys!=val.sample_key.astype(str).tolist():raise RuntimeError('Val order differs')
            scores=pad_score(z);threshold=threshold_from_val(y,scores)
            vm=metrics(y,scores,threshold,len(test))
            improved=active and (vm['acer']<best_acer-METRIC_TIE_EPS or
                (abs(vm['acer']-best_acer)<=METRIC_TIE_EPS and vm['auc']>best_auc+METRIC_TIE_EPS))
            if active:
                if improved:
                    best_acer=vm['acer'];best_auc=vm['auc'];best_epoch=epoch;no_improve=0
                    best_checkpoint={'run_id':name,'epoch':epoch,'config':cfg,
                        'val_acer':best_acer,'val_auc':best_auc,
                        'state_dict':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()}}
                    save_torch(best_path,best_checkpoint)
                no_improve=advance_patience(no_improve,improved,epoch,active)
            row={'epoch':epoch,'phase':'warmup' if warmup else 'active','lr':optimizer.param_groups[0]['lr'],
                 'eligible_for_final_checkpoint':active,'val_acer':vm['acer'],'val_auc':vm['auc'],
                 'val_eer':vm['eer'],'val_threshold':threshold,'best_epoch':best_epoch,
                 'no_improve_count':no_improve,'clean_ce_weight':1.0 if warmup else spec['clean_ce_weight'],
                 'spectral_ce_weight':0.0 if warmup else spec['spectral_ce_weight'],
                 'method':'CLEAN_WARMUP' if warmup else spec['method'],
                 **{k:v/seen for k,v in totals.items()},
                 **{f'{label}_count':band_count[i] for i,label in enumerate(('low','mid','high'))},
                 **{f'{label}_fraction':band_count[i]/seen for i,label in enumerate(('low','mid','high'))}}
            row['harmful_count']=harmful_count_total
            if spec['method']=='HARMFUL':
                row['harmful_ce_batch_mean']=row['harmful_ce']
                row['harmful_ce']=harmful_ce_sum_total/harmful_count_total if harmful_count_total else 0.
            if active and spec['method']!='CLEAN':diagnostics.append(dict(row))
            stop=epoch==MAX_EPOCHS or patience_stop_reached(epoch,no_improve)
            row['stop_reached']=stop;history.append(row)
            scheduler.step()
            state={'run_id':name,'config':cfg,'epoch':epoch,
                'model':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
                'scaler':scaler.state_dict(),'best_epoch':best_epoch,'best_acer':best_acer,'best_auc':best_auc,
                'no_improve':no_improve,'initial_state_sha256':INITIAL_STATE_SHA,'rng':snapshot_rng(gain_rng),'best_checkpoint':best_checkpoint,
                'history':history,'diagnostics':diagnostics}
            save_torch(last_path,state)
            pd.DataFrame(history).to_csv(folder/'training_history.csv',index=False)
            if spec['method']!='CLEAN':pd.DataFrame(diagnostics).to_csv(folder/'spectral_diagnostics.csv',index=False)
            print(name,'epoch',epoch,'Full Test-as-Val ACER',vm['acer'],'best',best_epoch,'patience',no_improve,flush=True)
            if stop:break
    if not best_path.is_file() or best_epoch<=spec['warmup_epochs']:
        raise RuntimeError(name+' has no eligible best checkpoint')
    best=torch.load(best_path,map_location='cpu',weights_only=True)
    if best['run_id']!=name or best['config']!=cfg or best['epoch']!=best_epoch:
        raise RuntimeError(name+' best checkpoint identity differs')
    model.load_state_dict(best['state_dict']);model.eval()
    keys,y,z=infer(model,val_dl)
    if keys!=val.sample_key.astype(str).tolist():raise RuntimeError('Frozen Val order differs')
    locked=threshold_from_val(y,pad_score(z))
    vm=metrics(y,pad_score(z),locked,len(test))
    if abs(vm['acer']-best['val_acer'])>1e-6 or abs(vm['auc']-best['val_auc'])>1e-6:
        raise RuntimeError('Best checkpoint does not reproduce Val selection metrics')
    write_json(folder/'metrics'/'val_metrics.json',vm)
    save_final_source_evaluation(folder,keys,y,z,locked)
    freeze={'run_id':name,'best_epoch':best_epoch,'locked_threshold':locked,'selection_set':'Full official Test-as-Val','checkpoint_sha256':sha(best_path),'source_metrics':vm,'val_metrics':vm,'manifest_sha256':MANIFEST_HASHES,
      'train_crop_factor':1.5,'val_crop_factor':1.5,'test_crop_factor':1.5,'lcc_crop_factor':1.5}
    freeze_path=folder/'evaluation_freeze.json'
    if freeze_path.is_file() and read_json(freeze_path)!=freeze:raise RuntimeError('Evaluation freeze differs')
    write_json(freeze_path,freeze)
    evaluate_lcc(model,folder,locked)
    write_json(done_path,{'run_id':name,'best_epoch':best_epoch,'locked_threshold':locked,
                          'initial_state_sha256':INITIAL_STATE_SHA})
    del model
    torch.cuda.empty_cache()

def assert_fairness_contract(configs,initial_hashes):
    if set(configs)!=set(RUNS) or set(initial_hashes)!=set(RUNS):raise RuntimeError('Fairness audit requires all three runs')
    if len(set(initial_hashes.values()))!=1 or not next(iter(initial_hashes.values())):raise RuntimeError('Initial-state SHA differs across methods')
    expected={'full_train.csv','full_test.csv'}
    for cfg in configs.values():
        if set(cfg['manifest_sha256'])!=expected or cfg['manifest_sha256']!=MANIFEST_HASHES:raise RuntimeError('Frozen manifest SHA differs across methods')
        if cfg['initial_state_sha256']!=initial_hashes[cfg['run_id']]:raise RuntimeError('Config/init SHA differs')
        if any(cfg[k]!=1.5 for k in ('train_crop_factor','val_crop_factor','test_crop_factor','lcc_crop_factor')):raise RuntimeError('Matched crop1.5 required')
    c=RUNS['C_FULL_crop15'];p=RUNS['P3_SF_FULL_crop15'];r=RUNS['R7_SC_FULL_crop15']
    if c['method']!='CLEAN' or c['warmup_epochs']!=0 or c['centers'] is not None or c['sigma'] is not None or (c['clean_ce_weight'],c['spectral_ce_weight'])!=(1.,0.):raise RuntimeError('Historical C mechanism drift')
    for spec in (p,r):
        if spec['centers']!=SCALED_CENTERS or (spec['clean_ce_weight'],spec['spectral_ce_weight'],spec['warmup_epochs'])!=(.75,.25,1):raise RuntimeError('Historical spectral mechanism drift')
    if p['method']!='WORST' or p['sigma']!=.10*(1.5/2.7) or r['method']!='HARMFUL' or r['sigma']!=.10:raise RuntimeError('SF/SC geometry mismatch')
    return True

def advance_patience(patience_counter,improved,epoch,eligible):
    # Checkpoint eligibility and improvement are determined before this rule.
    if improved or epoch<PATIENCE_START_EPOCH:return 0
    if not eligible:return patience_counter
    return patience_counter+1

def patience_stop_reached(epoch,patience_counter):
    return epoch>=PATIENCE_START_EPOCH and patience_counter>=EARLY_STOP_PATIENCE

def phase_patience_sanity_checks():
    for spec in RUNS.values():
        counter=0;stopped=None
        for epoch in range(1,MAX_EPOCHS+1):
            eligible=epoch>spec['warmup_epochs']
            counter=advance_patience(counter,False,epoch,eligible)
            if epoch<9:assert counter==0
            if patience_stop_reached(epoch,counter):stopped=epoch;break
        assert stopped==11
    counter=advance_patience(2,True,9,True)
    assert counter==0
    for epoch in (10,11,12):counter=advance_patience(counter,False,epoch,True)
    assert counter==3 and patience_stop_reached(12,counter)
    assert advance_patience(0,True,5,True)==0
    assert not patience_stop_reached(8,3)
    # Resuming committed epoch8 vs uninterrupted final-phase counts gives identical results.
    first=advance_patience(0,False,9,True);resumed=advance_patience(first,False,10,True)
    assert resumed==2 and advance_patience(resumed,False,11,True)==3
    return {'patience_start_epoch':9,'early_stop_patience':3,'earliest_possible_stop_epoch':11,
            'all_methods_identical':True,'improvement_resets_counter':True,'epoch_boundary_counter_replay':True,
            'best_selection_not_delayed':True}


def reference_metrics(y3,logits,candidate_n):
    y=(np.asarray(y3,dtype=int)!=0).astype(int)  # Reference 0=real, 1=attack.
    z=torch.as_tensor(np.asarray(logits));p=F.softmax(z,dim=-1)[:,1].cpu().numpy();pred=z.argmax(dim=1).cpu().numpy()
    real=y==0;attack=y==1
    if not real.any() or not attack.any():raise ValueError('Reference metrics need both classes')
    fpr,tpr,_=roc_curve(y,p,pos_label=1);idx=int(np.nanargmin(np.abs((1-tpr)-fpr)))
    apcer=float((pred[real]==1).mean());bpcer=float((pred[attack]==0).mean())
    return {'auc':float(roc_auc_score(y,p)),'eer':float(min(fpr[idx],1-tpr[idx])),
      'apcer':apcer,'bpcer':bpcer,'acer':.5*(apcer+bpcer),'global_argmax_accuracy':float((pred==y).mean()),
      'candidate_n':int(candidate_n),'scored_n':len(y),'coverage':len(y)/candidate_n,
      'convention':'Reference variable names: APCER=real->attack; BPCER=attack->real; argmax logits; ROC attack probability'}


## Initialization, method and phase-aware patience checks
C is eligible from epoch1; P3/R7 clean warmup epoch1 is diagnostic only. Patience starts at epoch9, patience3, earliest stop11; best checkpoint selection remains active earlier.


In [ ]:
# Actual initialization/manifest fairness for all three runs, before any training.
if (CROP_SCALE,TRAIN_CROP_FACTOR,EVAL_CROP_FACTOR)!=(1.5,1.5,1.5):raise RuntimeError('All phases require matched crop1.5')
if (DATA_SEED,GAIN_RANGE,PATIENCE_START_EPOCH,EARLY_STOP_PATIENCE,WARMUP_EPOCHS)!=(43,(.65,.90),9,3,1):raise RuntimeError('Frozen scale controls changed')
for filename,digest in MANIFEST_HASHES.items():
    if sha(MANIFEST_SOURCES[filename])!=digest or sha(out/'manifests'/filename)!=digest:raise RuntimeError('Frozen manifest changed since setup')
FAIR_CONFIGS={name:run_config(name,spec) for name,spec in RUNS.items()}
assert_fairness_contract(FAIR_CONFIGS,INITIAL_HASHES)
write_json(out/'fairness_checks.json',{'passed':True,'configs':FAIR_CONFIGS,'initial_state_sha256':INITIAL_HASHES,'manifest_sha256':MANIFEST_HASHES,
  'historical_C_eligibility':'notebook04 warmup_epochs0, epoch1 eligible','historical_P3_R7_eligibility':'notebook04/09 warmup_epoch1 excluded from best/patience'})

def mechanism_sanity_checks():
    class TinyPAD(nn.Module):
        def __init__(self):
            super().__init__();self.w=nn.Parameter(torch.tensor(1.));self.bn=nn.BatchNorm1d(3);self.calls=[];self.last_z=None
        def forward(self,x,aux=False):
            self.calls.append((self.training,torch.is_grad_enabled(),len(x)))
            a=self.bn(x.mean((2,3)));d=(a[:,0]-a[:,1])*self.w;z=torch.stack((d,-d),1);self.last_z=z
            return (z,d[:,None].expand(-1,11),d[:,None].expand(-1,5),d[:,None].expand(-1,40)) if aux else z
    global use_amp,select_worst_view
    old_amp=use_amp;original_selector=select_worst_view;use_amp=False
    try:
        with torch.random.fork_rng(devices=[]):
            x=torch.rand(4,3,80,80);y=torch.tensor([0,1,0,1]);labels=torch.zeros(4,dtype=torch.long);attrs=torch.zeros(4,40)
            masks=radial_masks(SCALED_CENTERS,P3_SF_SIGMA,device=torch.device('cpu'))
            p=TinyPAD().train();gain=torch.Generator().manual_seed(GAIN_SEED);state=gain.get_state().clone()
            original_selector(p,x,y,masks,gain)
            assert p.training and int(p.bn.num_batches_tracked)==0 and all(not train and not grad for train,grad,_ in p.calls)
            gain.set_state(state);first=original_selector(TinyPAD().train(),x,y,masks,gain)[1]
            gain.set_state(state);second=original_selector(TinyPAD().train(),x,y,masks,gain)[1]
            assert torch.equal(first,second)
            def forbidden(*args):raise AssertionError('C/warmup entered spectral selection')
            select_worst_view=forbidden
            for name,active in (('C_FULL_crop15',True),('P3_SF_FULL_crop15',False),('R7_SC_FULL_crop15',False)):
                model=TinyPAD().train();loss,d,b,_=training_terms(model,x,y,labels,labels,attrs,RUNS[name],None,gain,active)
                assert len(model.calls)==1 and not b and d['spectral_ce']==0. and abs(loss.item()-d['clean_ce']-d['aux_loss'])<1e-6
            # Controlled detached selection metadata, no model/data fitting.
            def chosen_with_gate(harmful):
                def choose(*args):
                    cm=torch.full((4,),2.);wm=torch.where(harmful,torch.full((4,),1.),cm)
                    return x.roll(1,0),torch.tensor([0,1,2,0]),torch.full((4,),.75),torch.zeros(4),cm,wm
                return choose
            for gate in (torch.zeros(4,dtype=torch.bool),torch.tensor([True,False,True,False])):
                select_worst_view=chosen_with_gate(gate)
                model=TinyPAD().train();loss,d,b,_=training_terms(model,x,y,labels,labels,attrs,RUNS['R7_SC_FULL_crop15'],masks,gain,True)
                assert d['harmful_count']==int(gate.sum()) and len(model.calls)==2
                assert abs(loss.item()-(.75*d['clean_ce']+.25*d['harmful_ce']+d['aux_loss']))<1e-6
                student_grad=torch.autograd.grad(loss,model.last_z,retain_graph=True)[0]
                if not gate.any():assert d['harmful_ce']==0. and torch.equal(student_grad,torch.zeros_like(student_grad))
                else:assert torch.equal(student_grad[~gate],torch.zeros_like(student_grad[~gate])) and student_grad[gate].abs().sum()>0
                loss.backward();assert model.w.grad is not None
            select_worst_view=original_selector
            model=TinyPAD().train();loss,d,_,_=training_terms(model,x,y,labels,labels,attrs,RUNS['P3_SF_FULL_crop15'],masks,gain,True)
            assert abs(loss.item()-(.75*d['clean_ce']+.25*d['worst_ce']+d['aux_loss']))<1e-6
            assert len([c for c in model.calls if c[1]])==2 and int(model.bn.num_batches_tracked)==2
            saved=snapshot_rng(gain);draw=torch.rand(4,generator=gain);box=bbox_jitter({'bbox_xyxy':[10,10,50,50]})
            restore_rng(saved,gain)
            assert torch.equal(draw,torch.rand(4,generator=gain)) and box==bbox_jitter({'bbox_xyxy':[10,10,50,50]})
    finally:select_worst_view=original_selector;use_amp=old_amp
    return {'C_clean_only_epoch1_eligible':True,'P3_R7_clean_warmup_excluded':True,'BN_safe_detached_hard_selection':True,
      'P3_reference_objective':True,'R7_harmful_denominator_and_differentiable_zero':True,'R7_no_harmful_clean_coefficient_075':True,
      'one_selected_gradient_view':True,'gain_and_bbox_rng_restore':True}

IMPLEMENTATION_CHECKS=mechanism_sanity_checks()
IMPLEMENTATION_CHECKS['phase_aware_patience']=phase_patience_sanity_checks()
write_json(out/'implementation_sanity_checks.json',IMPLEMENTATION_CHECKS)
print('[Fairness/Sanity] Passed:',IMPLEMENTATION_CHECKS,flush=True)


## Reports and exports
Project threshold-based metrics and reference argmax/probability metrics are separate. Root and per-run ZIP downloads are written after each completed run.


In [ ]:
REFERENCE_AUDIT={'repository': 'https://github.com/Inria-CENATAV-Tec/Assessing-Efficient-FAS-CVPR2024', 'commit': '38233cd3eb25a69d4fcc6aa681d49cddcd635907', 'source_sha256': {'README.md': '61125717b9e69efb189f1dede29b952eee9922fd52e4e7453daa6ba788d16bb4', 'eval_protocol.py': '1b299f47393f7ae46292ab7f029989854a688b156cfa1b5080f65e9b8e2b143f', 'datasets/celeba_spoof.py': 'b55cedf13b7ad14ec23a57c6bb74c51f128afd4738f679e437e0b6e6d0f8e62b', 'datasets/lcc_fasd.py': '7bab548a4b3f63f81236d8aedc5d76dc2d4d8241af2a58f6247825a0f74df61c', 'datasets/database.py': 'b17309390478750c94d635193f75567775ee3eabbde2a9bb3550628d9915960b', 'utils.py': '891321f3738c28d032114506611b7e1bb6ce1cd320a5edff23da56a0736bdb50', 'trainer.py': '7bfc7c07777f3d8458d808b18cf5883536070f115ca40008c5cba4cd3e1d5923'}, 'source_urls': {'README.md': 'https://github.com/Inria-CENATAV-Tec/Assessing-Efficient-FAS-CVPR2024/blob/38233cd3eb25a69d4fcc6aa681d49cddcd635907/README.md', 'eval_protocol.py': 'https://github.com/Inria-CENATAV-Tec/Assessing-Efficient-FAS-CVPR2024/blob/38233cd3eb25a69d4fcc6aa681d49cddcd635907/eval_protocol.py', 'datasets/celeba_spoof.py': 'https://github.com/Inria-CENATAV-Tec/Assessing-Efficient-FAS-CVPR2024/blob/38233cd3eb25a69d4fcc6aa681d49cddcd635907/datasets/celeba_spoof.py', 'datasets/lcc_fasd.py': 'https://github.com/Inria-CENATAV-Tec/Assessing-Efficient-FAS-CVPR2024/blob/38233cd3eb25a69d4fcc6aa681d49cddcd635907/datasets/lcc_fasd.py', 'datasets/database.py': 'https://github.com/Inria-CENATAV-Tec/Assessing-Efficient-FAS-CVPR2024/blob/38233cd3eb25a69d4fcc6aa681d49cddcd635907/datasets/database.py', 'utils.py': 'https://github.com/Inria-CENATAV-Tec/Assessing-Efficient-FAS-CVPR2024/blob/38233cd3eb25a69d4fcc6aa681d49cddcd635907/utils.py', 'trainer.py': 'https://github.com/Inria-CENATAV-Tec/Assessing-Efficient-FAS-CVPR2024/blob/38233cd3eb25a69d4fcc6aa681d49cddcd635907/trainer.py'}, 'population_protocol': 'Full official CelebA Test and pooled full LCC, confirmed by user; no further population audit requested', 'reference_label_mapping': {'real': 0, 'attack': 1}, 'our_binary_mapping': {'real': 0, 'attack': 1}, 'label_inversion_required': False, 'score': 'softmax(two PAD logits)[:,1], attack positive', 'classification': 'argmax raw logits (tie chooses class 0)', 'eer': 'min(FPR,FNR) at first nanargmin(abs(FNR-FPR)); no interpolation', 'apcer_reference': 'FP / real_count (real classified attack), corresponds to project BPCER at argmax threshold', 'bpcer_reference': 'FN / attack_count (attack classified real), corresponds to project APCER at argmax threshold', 'accuracy': 'We additionally report global argmax accuracy; reference averages per-batch accuracies, which can differ on the final partial batch', 'project_metrics': 'Preserved real-positive logit difference, source-calibrated threshold, ISO rate names, averaged FPR/FNR EER', 'preprocessing': 'Reference tight crops, RGB 128x128 normalized; project frozen SCRFD crop1.5, BGR 80x80 [0,1]. Impact is not measured.', 'comparison_caveat': 'Same full split scope as confirmed by user; scored population can differ with localization/preprocessing. Metric-compatible comparison is not identical preprocessing.', 'reference_snapshot_caveat': 'LCC registry entries are commented at this pinned snapshot; constants are published README results, not reproduced runs.', 'aenet_caveat': 'README imports AENET results; exact evaluator identity for those imported results is not established', 'units': 'AUC fraction; computed rates fractions; published README EER/APCER/BPCER/ACER percent; params millions; GFLOPs as published'}
REFERENCE_BENCHMARKS={'source': 'https://github.com/Inria-CENATAV-Tec/Assessing-Efficient-FAS-CVPR2024/blob/38233cd3eb25a69d4fcc6aa681d49cddcd635907/README.md', 'rows': [{'dataset': 'CelebA', 'model': 'MN3_large', 'auc': 0.998, 'eer_percent': 2.26, 'apcer_percent': 0.69, 'bpcer_percent': 6.92, 'acer_percent': 3.8, 'params_m': 3.02, 'gflops_published': 0.15}, {'dataset': 'CelebA', 'model': 'AENET', 'auc': 0.999, 'eer_percent': 1.12, 'apcer_percent': 0.23, 'bpcer_percent': 6.27, 'acer_percent': 3.25, 'params_m': 11.22, 'gflops_published': 3.64}, {'dataset': 'CelebA', 'model': 'MN3_large_075', 'auc': 0.997, 'eer_percent': 2.79, 'apcer_percent': 1.21, 'bpcer_percent': 5.44, 'acer_percent': 3.32, 'params_m': 1.86, 'gflops_published': 0.1}, {'dataset': 'CelebA', 'model': 'MN3_small', 'auc': 0.994, 'eer_percent': 3.84, 'apcer_percent': 1.47, 'bpcer_percent': 8.63, 'acer_percent': 5.05, 'params_m': 1.0, 'gflops_published': 0.04}, {'dataset': 'CelebA', 'model': 'MN3_small_075', 'auc': 0.991, 'eer_percent': 4.74, 'apcer_percent': 1.62, 'bpcer_percent': 10.55, 'acer_percent': 6.09, 'params_m': 0.6, 'gflops_published': 0.03}, {'dataset': 'LCC', 'model': 'MN3_large', 'auc': 0.921, 'eer_percent': 16.13, 'apcer_percent': 17.26, 'bpcer_percent': 15.4, 'acer_percent': 16.33, 'params_m': 3.02, 'gflops_published': 0.15}, {'dataset': 'LCC', 'model': 'AENET', 'auc': 0.868, 'eer_percent': 20.91, 'apcer_percent': 12.52, 'bpcer_percent': 32.7, 'acer_percent': 22.61, 'params_m': 11.22, 'gflops_published': 3.64}, {'dataset': 'LCC', 'model': 'MN3_large_075', 'auc': 0.892, 'eer_percent': 19.42, 'apcer_percent': 28.34, 'bpcer_percent': 12.18, 'acer_percent': 20.26, 'params_m': 1.86, 'gflops_published': 0.1}, {'dataset': 'LCC', 'model': 'MN3_small', 'auc': 0.889, 'eer_percent': 18.7, 'apcer_percent': 14.79, 'bpcer_percent': 24.6, 'acer_percent': 19.69, 'params_m': 1.0, 'gflops_published': 0.04}, {'dataset': 'LCC', 'model': 'MN3_small_075', 'auc': 0.879, 'eer_percent': 21.07, 'apcer_percent': 22.77, 'bpcer_percent': 19.3, 'acer_percent': 21.04, 'params_m': 0.6, 'gflops_published': 0.03}]}

# Read-only historical context. Never used by training, selection or calibration.
SCALE100K_REFERENCES={'C':.848273,'P3_SF':.851978,'R7_SC':.867844}
SCORE_SPLITS=('test','lcc_training','lcc_development','lcc_evaluation','lcc_combined')
TABLE_COLUMNS=('selected_epoch','test_auc','test_eer','test_acer','lcc_evaluation_auc','lcc_evaluation_eer','lcc_evaluation_hter',
 'lcc_combined_auc','lcc_combined_eer','lcc_combined_hter','lcc_combined_tpr_at_fpr_1pct','lcc_combined_detector_coverage')
EVALUATION_PROTOCOL={'training_budget_note':'The full CelebA-Spoof training set is approximately 500K samples. Therefore the full-scale training budget uses 12 epochs with milestones [4,8], while preserving the same three-phase SGD schedule structure.','train':'Entire eligible official CelebA Train, no sampling',
 'selection':'Full official CelebA Test-as-Val at every epoch; C eligible epoch1, P3/R7 epoch1 excluded',
 'final_source':'Reload best; re-infer full usable Test once; recalibrate/freeze source threshold before LCC',
 'independent_test':False,'lcc':'Full physical training/development/evaluation and pooled Combined; no LCC tuning',
 'patience_start_epoch':9,'early_stop_patience':3,'max_epochs':12,'earliest_possible_stop_epoch':11,
 'historical_protocol_identical':False,'reference_comparison':'Full Test/full LCC scope; differing preprocessing permitted by user',
 'comparison_role':'Descriptive one-seed confirmatory frozen-design comparison; no PASS/FAIL survival gate'}
write_json(out/'evaluation_protocol.json',EVALUATION_PROTOCOL)
write_json(out/'reference_metric_compatibility_audit.json',REFERENCE_AUDIT)
write_json(out/'reference_benchmark_table.json',REFERENCE_BENCHMARKS)

def clean_complexity():
    model,_=init_model({'init':'pretrained','copied':True});model.eval()
    aux_names=('spoof_type_head.','lighting_head.','attributes_head.')
    total=sum(p.numel() for p in model.parameters());trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
    deployed=sum(p.numel() for name,p in model.named_parameters() if not name.startswith(aux_names))
    result={'input_shape':[1,3,80,80],'total_params':total,'trainable_params':trainable,
      'deployment_params':deployed,'params_m':deployed/1e6,'deployment':'MiniFASNetV2 feature trunk + binary PAD head only',
      'counting_method':'torch.profiler CPU with_flops=True on clean forward; Conv2d/Linear supported operator FLOPs; MACs=FLOPs/2',
      'scope':'Conv/Linear arithmetic only; normalization/activation/pooling costs not included. Published reference counting convention may differ.',
      'spectral_training_included':False,'same_inference_architecture_for_all_runs':True,'macs':None,'gflops':None}
    try:
        with torch.no_grad(),torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU],record_shapes=True,with_flops=True) as prof:
            model(torch.zeros(1,3,80,80))
        flops=sum(int(event.flops) for event in prof.key_averages())
        if not flops:raise RuntimeError('Profiler returned no supported FLOPs')
        result.update(macs=flops/2,gflops=flops/1e9,flops=flops,measurement_status='computed')
    except Exception as error:result.update(measurement_status='unavailable',reason=str(error))
    del model
    return {'shared_clean_inference':result,'runs':{name:dict(result) for name in RUNS},'reference_values':REFERENCE_BENCHMARKS['rows']}
COMPLEXITY=clean_complexity()
write_json(out/'complexity_report.json',COMPLEXITY)
print('[Complexity]',COMPLEXITY['shared_clean_inference'],flush=True)

def result_row(name,folder):
    row={'run_id':name,'status':'pending'}
    if not (folder/'completion.json').is_file():return row
    row.update(status='complete',selected_epoch=read_json(folder/'completion.json')['best_epoch'])
    for split in SCORE_SPLITS:
        for key,value in read_json(folder/'metrics'/(split+'_metrics.json')).items():row[split+'_'+key]=value
    return row

def value_text(value):
    if isinstance(value,(int,float)):return f'{value:.6f}'
    return str(value) if value is not None else 'pending'

def export_summary():
    rows=[result_row(name,out/'runs'/name) for name in RUNS]
    measured={r['run_id']:r for r in rows if r['status']=='complete'}
    project={};reference={}
    for name in measured:
        folder=out/'runs'/name
        project[name]={s:read_json(folder/'metrics'/(s+'_metrics.json')) for s in SCORE_SPLITS}
        reference[name]={s:read_json(folder/'metrics'/(s+'_reference_metrics.json')) for s in SCORE_SPLITS}
    comparisons={}
    for label,a,b in (('P3_minus_C','P3_SF_FULL_crop15','C_FULL_crop15'),('R7_minus_C','R7_SC_FULL_crop15','C_FULL_crop15'),('R7_minus_P3','R7_SC_FULL_crop15','P3_SF_FULL_crop15')):
        comparisons[label]={metric:measured[a]['lcc_combined_'+metric]-measured[b]['lcc_combined_'+metric] for metric in ('auc','eer','hter','tpr_at_fpr_1pct')} if a in measured and b in measured else None
    ranking=[r['run_id'] for r in sorted(measured.values(),key=lambda r:-r['lcc_combined_auc'])]
    historical=sorted(SCALE100K_REFERENCES,key=SCALE100K_REFERENCES.get,reverse=True)
    comparison_rows=[]
    for row in rows:
        combined_row=dict(row)
        for split,mm in reference.get(row['run_id'],{}).items():
            for key in ('auc','eer','apcer','bpcer','acer','global_argmax_accuracy'):combined_row['reference_'+split+'_'+key]=mm[key]
        shared=COMPLEXITY['shared_clean_inference']
        combined_row.update(deployment_params_m=shared['params_m'],total_params=shared['total_params'],trainable_params=shared['trainable_params'],clean_inference_macs=shared['macs'],clean_inference_gflops=shared['gflops'])
        comparison_rows.append(combined_row)
    pd.DataFrame(comparison_rows).to_csv(out/'fullscale_comparison.csv',index=False)
    write_json(out/'fullscale_project_metrics.json',{'runs':project,'pairwise_combined_deltas':comparisons,'descriptive_ordering':ranking,'scale100k_reporting_only':SCALE100K_REFERENCES})
    write_json(out/'fullscale_reference_compatible_metrics.json',{'runs':reference,'audit_file':'reference_metric_compatibility_audit.json','units':'AUC and rates fractions; displayed rates percent'})
    report=['# Fullscale C / P3_SF / R7_SC — crop1.5','',
      'Full eligible official Train → full official Test-as-Val → full LCC train/dev/eval/Combined. Test-as-Val is a selection set, not an untouched independent Test.',
      'The full CelebA-Spoof training set is approximately 500K samples. Therefore the full-scale training budget uses 12 epochs with milestones [4,8], while preserving the same three-phase SGD schedule structure.',
      'New phase-aware patience: patience_start_epoch=9, early_stop_patience=3, earliest stop11, max12; validation and best selection are not delayed. This differs from older historical scale100K training protocols.',
      'C eligible epoch1; P3/R7 epoch1 clean warmup excluded. All methods retain SGD .005/.9/5e-4, milestones[4,8], gamma.2, batch256 AMP, TRAIN_SEED100, DATA_SEED43.',
      'Frozen P3_SF scaled centers and sigma; R7_SC same centers and sigma.10; clean/spectral .75/.25, clean-only auxiliary losses. No mechanism tuning or fullscale survival gate.',
      'Reload best and re-infer full usable Test; freeze its calibrated threshold/checkpoint SHA before LCC. Reference metrics use the same logits and require no additional inference.','',
      '## Dataset accounting','', '```json',json.dumps(DATA_COUNTS,indent=2),'```','',
      '## Project metrics (source-calibrated threshold)','',
      '| Run | '+' | '.join(TABLE_COLUMNS)+' |','|---|'+'---:|'*len(TABLE_COLUMNS)]
    for row in rows:report.append('| '+row['run_id']+' | '+' | '.join(value_text(row.get(k)) for k in TABLE_COLUMNS)+' |')
    report+=['','## Full Test and full LCC comparison with the published reference','',
      'Same full split scope as confirmed by the user; preprocessing differs. Our full Test numbers reuse the checkpoint-selection set. Published benchmark values are copied from the pinned README, not rerun here.',
      'Reference APCER variable means real→attack; BPCER means attack→real. This table preserves those names. Reference EER=min(FPR,FNR) at nearest crossing; project EER averages them. Reference rates use binary argmax, independently of our calibrated threshold.',
      'Params are deployment parameters (millions); our GFLOPs cover profiler-supported Conv/Linear operations at 80x80. Published reference input is 128x128 and counting conventions may differ. FFT/selection is training-only.',
      'Full LCC comparison uses Combined. Physical Evaluation metrics are additionally reported above.','',
      '| Dataset | Model | AUC | EER % | APCER % | BPCER % | ACER % | Params M | GFLOPs |',
      '|---|---|---:|---:|---:|---:|---:|---:|---:|']
    for b in REFERENCE_BENCHMARKS['rows']:
        report.append('| '+b['dataset']+' | '+b['model']+' (published) | '+' | '.join(value_text(b[k]) for k in ('auc','eer_percent','apcer_percent','bpcer_percent','acer_percent','params_m','gflops_published'))+' |')
    complexity=COMPLEXITY['shared_clean_inference']
    for dataset,split in (('CelebA Test-as-Val','test'),('LCC Combined','lcc_combined')):
        for name in RUNS:
            m=reference.get(name,{}).get(split,{})
            values=[m.get('auc')]+[100*m[k] if k in m else None for k in ('eer','apcer','bpcer','acer')]+[complexity['params_m'],complexity['gflops']]
            report.append('| '+dataset+' | '+name+' | '+' | '.join(value_text(v) for v in values)+' |')
    report+=['','## Physical LCC metrics and coverage','']
    for split in SCORE_SPLITS:
        cols=('auc','eer','apcer','bpcer','acer','hter','binary_accuracy','candidate_n','scored_n','detector_coverage')
        report+=['### '+split+' (project)','', '| Run | '+' | '.join(cols)+' |','|---|'+'---:|'*len(cols)]
        for row in rows:report.append('| '+row['run_id']+' | '+' | '.join(value_text(row.get(split+'_'+k)) for k in cols)+' |')
        report.append('')
    report+=['## Descriptive fullscale vs 100K context','',
      'Pairwise Combined deltas (first minus second): '+json.dumps(comparisons)+'.',
      'Completed fullscale Combined-AUC order: '+str(ranking)+'. Full ordering pending until all three complete.',
      'Read-only100K AUC: '+json.dumps(SCALE100K_REFERENCES)+'; ordering '+str(historical)+'. No PASS/FAIL gate and no statistical superiority claim.',
      'Training diagnostics remain in each run/training_history.csv and spectral_diagnostics.csv (P3/R7): clean/worst/harmful CE, harmful counts/fraction, margin/drop, flip, gains, selected bands.','',
      '## Limitations','',
      '- One seed; descriptive comparison, no significance or multi-seed claim.',
      '- CelebA full Test is used for selection and threshold calibration; its final metrics are not independent Test evidence.',
      '- LCC has been repeatedly inspected during method development. This is a confirmatory experiment with a frozen design, not a fresh independent benchmark.',
      '- Official identity overlap is disclosed and retained; no new population filtering or split construction.',
      '- Reference and project preprocessing/localization differ; the size of their effect has not been measured. Scored coverage is explicit.',
      '- Published reference values, including imported AENET, are not reproduced runs; the metric audit and complexity scope document the conventions.',
      '- Multi-seed confirmation and additional independent validation remain future work.','',
      '## Artifacts','', 'Dataset counts, manifest copies/hashes, init equality, config, best/last checkpoint, evaluation_freeze.json, both metric layers, complexity, histories and predictions are retained.']
    (out/'fullscale_report.md').write_text('\n'.join(report)+'\n')
    return rows

def export_run_zip(name):
    folder=out/'runs'/name;run_zip=out.parent/(name+'.zip')
    with zipfile.ZipFile(run_zip,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
        for file in sorted(folder.rglob('*')):
            if file.is_file():archive.write(file,(Path('runs')/name/file.relative_to(folder)).as_posix())
        for file in sorted(out.glob('*')):
            if file.is_file():archive.write(file,file.name)
        for file in sorted((out/'manifests').glob('*.csv')):archive.write(file,(Path('manifests')/file.name).as_posix())
    display(FileLink(str(run_zip)))

def export_overall_zip():
    zip_path=out.parent/'18_fullscale_C_P3SF_R7SC_crop15.zip'
    print('[Export]',zip_path,flush=True)
    with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
        for file in sorted(out.rglob('*')):
            if file.is_file():archive.write(file,(Path(out.name)/file.relative_to(out)).as_posix())
    display(FileLink(str(zip_path)))

export_summary();export_overall_zip()


## Run selected experiments
Full Train and full Test-as-Val make this more expensive than Train100K/Val15K. Batch logs appear every LOG_EVERY_BATCHES and at inference boundaries. Resume is at committed epoch boundaries.


In [ ]:
selected=select_runs(RUN_FILTER)
print('[Runner] Selected runs:',selected,flush=True)
# Recheck fairness before starting any selected GPU training.
assert_fairness_contract({name:run_config(name,spec) for name,spec in RUNS.items()},INITIAL_HASHES)
for name in selected:
    try:run_one(name,RUNS[name])
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError('Frozen batch256 ran out of GPU memory; stop instead of changing the recipe') from exc
    export_summary()
    export_run_zip(name)  # Export completed run and current reports before starting another run.
    export_overall_zip()
print('Finished selected fullscale runs:',selected,flush=True)
for name in selected:display(FileLink(str(out.parent/(name+'.zip'))))
display(FileLink(str(out.parent/'18_fullscale_C_P3SF_R7SC_crop15.zip')))
